# DataVisualisation-TP2025 — TP6

[Teacher assignment](https://github.com/ahmed3991/DataVisualisation-TP2025/blob/main/TP6/TP6.md)

Run the cells in order. Private credentials are read from an ignored .env file or hidden input. Read EXPLICATION_AR.md for the Arabic explanation.

This notebook needs your Supabase PostgreSQL connection. It loads the complete instructor-provided Northwind database, then runs and tests dbt models. It replaces this coursework snapshot on reruns.

## Install required libraries if needed

In [1]:
import importlib.util
import subprocess
import sys
packages = {'pandas': 'pandas', 'requests': 'requests', 'psycopg2': 'psycopg2-binary', 'dotenv': 'python-dotenv', 'dlt': 'dlt[postgres]', 'dbt': 'dbt-postgres'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])


## Companion files

In [2]:
from pathlib import Path
# Recreate the companion source/configuration files; no passwords are embedded.
support_files = {'connection.py': '"""Read a private PostgreSQL connection for both dlt and dbt."""\nimport os\nfrom getpass import getpass\nfrom pathlib import Path\nfrom urllib.parse import quote, urlparse, unquote, parse_qs\nfrom dotenv import load_dotenv\n\nROOT = Path(__file__).resolve().parent\n\ndef configure_connection():\n    load_dotenv(ROOT / ".env")\n    url = os.environ.get("DATABASE_URL")\n    if not url:\n        password = os.environ.get("DATABASE_PASSWORD") or getpass("Supabase database password (hidden): ")\n        host = os.environ.get("SUPABASE_HOST") or input("Supabase session pooler host: ").strip()\n        user = os.environ.get("SUPABASE_USER") or input("Supabase database user: ").strip()\n        port = os.environ.get("SUPABASE_PORT", "5432")\n        database = os.environ.get("SUPABASE_DATABASE", "postgres")\n        url = f"postgresql://{quote(user, safe=\'\')}:{quote(password, safe=\'\')}@{host}:{port}/{database}?sslmode=require"\n        os.environ["DATABASE_URL"] = url\n    parsed = urlparse(url)\n    if parsed.scheme not in {"postgres", "postgresql"}:\n        raise ValueError("A PostgreSQL connection URL is required.")\n    os.environ.update({"PGHOST": parsed.hostname or "localhost", "PGPORT": str(parsed.port or 5432),\n        "PGUSER": unquote(parsed.username or ""), "PGPASSWORD": unquote(parsed.password or ""),\n        "PGDATABASE": unquote(parsed.path.lstrip("/")),\n        "PGSSLMODE": parse_qs(parsed.query).get("sslmode", ["require"])[0]})\n    return url\n', 'sqlite_pipeline.py': '"""Load all seven required Northwind tables into Supabase staging with dlt."""\nimport os\nimport sqlite3\nfrom pathlib import Path\nimport requests\nimport dlt\nfrom connection import configure_connection\n\nROOT = Path(__file__).resolve().parent\nSOURCE_URL = "https://raw.githubusercontent.com/ahmed3991/DataVisualisation-TP2025/main/TP6/northwind.db"\nTABLES = {"Customers": "customers", "Orders": "orders", "Employees": "employees",\n          "Products": "products", "Categories": "categories", "Suppliers": "suppliers",\n          "Order Details": "order_details"}\nDECIMAL_COLUMNS = {"Products": ["UnitPrice"], "Orders": ["Freight"],\n                   "Order Details": ["UnitPrice", "Discount"]}\n\ndef ensure_database():\n    path = ROOT / "northwind.db"\n    if not path.exists():\n        response = requests.get(SOURCE_URL, timeout=120)\n        response.raise_for_status()\n        path.write_bytes(response.content)\n    return path\n\ndef table_rows(source_table):\n    with sqlite3.connect(ensure_database()) as connection:\n        cursor = connection.execute(f\'SELECT * FROM "{source_table}"\')\n        names = [column[0] for column in cursor.description]\n        while rows := cursor.fetchmany(5000):\n            yield [dict(zip(names, row)) for row in rows]\n\ndef load_sqlite_data():\n    os.environ.setdefault("DLT_TELEMETRY", "false")\n    url = configure_connection()\n    pipeline = dlt.pipeline(pipeline_name="northwind_sqlite_copy", destination="postgres",\n                           dataset_name="staging", pipelines_dir=str(ROOT / ".dlt-state"))\n    for source, destination in TABLES.items():\n        # SQLite NUMERIC values may arrive as either int or float. Declare the\n        # intended decimal type so dlt keeps all prices in the same column.\n        columns = {name: {"data_type": "decimal", "precision": 18, "scale": 6}\n                   for name in DECIMAL_COLUMNS.get(source, [])}\n        resource = dlt.resource(table_rows(source), name=destination,\n                                write_disposition="replace", columns=columns)\n        # Preserve binary images using standard inserts for these two tiny tables.\n        # Other tables use CSV/COPY to avoid many small network INSERT calls.\n        file_format = "insert_values" if source in {"Employees", "Categories"} else "csv"\n        info = pipeline.run(resource, credentials=url, loader_file_format=file_format)\n        info.raise_on_failed_jobs()\n        print("Loaded:", source)\n    return pipeline\n\nif __name__ == "__main__":\n    load_sqlite_data()\n', 'northwind_T/dbt_project.yml': "name: northwind_T\nversion: '1.0.0'\nconfig-version: 2\nprofile: northwind_T\nmodel-paths: [models]\nflags:\n  send_anonymous_usage_stats: false\nmodels:\n  northwind_T:\n    staging:\n      +materialized: view\n    marts:\n      +materialized: table\n", 'northwind_T/package-lock.yml': 'packages:\n  - name: dbt_utils\n    package: dbt-labs/dbt_utils\n    version: 1.4.1\nsha1_hash: 8b27037b26f3f630c6661194d2470e720c49f6ee\n', 'northwind_T/packages.yml': 'packages:\n  - package: dbt-labs/dbt_utils\n    version: 1.4.1\n', 'northwind_T/profiles.yml': 'northwind_T:\n  target: dev\n  outputs:\n    dev:\n      type: postgres\n      host: "{{ env_var(\'PGHOST\') }}"\n      port: "{{ env_var(\'PGPORT\') | int }}"\n      user: "{{ env_var(\'PGUSER\') }}"\n      password: "{{ env_var(\'PGPASSWORD\') }}"\n      dbname: "{{ env_var(\'PGDATABASE\') }}"\n      sslmode: "{{ env_var(\'PGSSLMODE\', \'require\') }}"\n      schema: northwind_warehouse\n      threads: 2\n      connect_timeout: 30\n', 'northwind_T/models/marts/dim_categories.sql': "select * from {{ ref('stg_categories') }}\n", 'northwind_T/models/marts/dim_customers.sql': "with first_orders as (\n    select customer_id, min(order_date) as first_order_date\n    from {{ ref('stg_orders') }} group by customer_id\n)\nselect c.*, o.first_order_date,\n       concat_ws(', ', c.address, c.city, c.region, c.country) as customer_full_address\nfrom {{ ref('stg_customers') }} c\nleft join first_orders o on c.customer_id = o.customer_id\n", 'northwind_T/models/marts/dim_employees.sql': "select *, first_name || ' ' || last_name as employee_full_name\nfrom {{ ref('stg_employees') }}\n", 'northwind_T/models/marts/dim_products.sql': "select p.*, c.category_name, s.company_name as supplier_name\nfrom {{ ref('stg_products') }} p\nleft join {{ ref('stg_categories') }} c on p.category_id = c.category_id\nleft join {{ ref('stg_suppliers') }} s on p.supplier_id = s.supplier_id\n", 'northwind_T/models/marts/dim_suppliers.sql': "select * from {{ ref('stg_suppliers') }}\n", 'northwind_T/models/marts/fct_orders.sql': "with totals as (\n    select order_id, sum(unit_price * quantity * (1 - discount)) as total_order_value\n    from {{ ref('stg_order_details') }} group by order_id\n)\nselect o.*, c.customer_pk, e.employee_pk, t.total_order_value,\n       case when o.shipped_date is not null then 'Shipped' else 'Pending' end as order_status_flag,\n       extract(epoch from (o.shipped_date - o.order_date)) / 86400.0 as shipping_delay_days\nfrom {{ ref('stg_orders') }} o\nleft join totals t on o.order_id = t.order_id\nleft join {{ ref('dim_customers') }} c on o.customer_id = c.customer_id\nleft join {{ ref('dim_employees') }} e on o.employee_id = e.employee_id\n", 'northwind_T/models/marts/fct_order_details.sql': "select d.order_detail_pk, o.order_pk, p.product_pk, c.customer_pk, e.employee_pk,\n       d.order_id, d.product_id, d.unit_price, d.quantity, d.discount,\n       d.unit_price * d.quantity * (1 - d.discount) as extended_price,\n       o.order_date::date as order_date, o.required_date::date as required_date,\n       o.shipped_date::date as shipped_date,\n       extract(epoch from (o.shipped_date - o.order_date)) / 86400.0 as shipping_delay_days\nfrom {{ ref('stg_order_details') }} d\nleft join {{ ref('stg_orders') }} o on d.order_id = o.order_id\nleft join {{ ref('dim_products') }} p on d.product_id = p.product_id\nleft join {{ ref('dim_customers') }} c on o.customer_id = c.customer_id\nleft join {{ ref('dim_employees') }} e on o.employee_id = e.employee_id\n", 'northwind_T/models/marts/schema.yml': "version: 2\nmodels:\n- name: dim_customers\n  description: Northwind dim_customers for the required data mart.\n  columns:\n  - name: customer_pk\n    description: Primary surrogate key.\n    data_tests:\n    - unique\n    - not_null\n  - name: customer_id\n    description: Customer id.\n  - name: company_name\n    description: Company name.\n  - name: contact_name\n    description: Contact name.\n  - name: contact_title\n    description: Contact title.\n  - name: address\n    description: Address.\n  - name: city\n    description: City.\n  - name: region\n    description: Region.\n  - name: postal_code\n    description: Postal code.\n  - name: country\n    description: Country.\n  - name: phone\n    description: Phone.\n  - name: fax\n    description: Fax.\n  - name: first_order_date\n    description: Earliest order timestamp for the customer.\n  - name: customer_full_address\n    description: Address, city, region and country.\n- name: dim_employees\n  description: Northwind dim_employees for the required data mart.\n  columns:\n  - name: employee_pk\n    description: Primary surrogate key.\n    data_tests:\n    - unique\n    - not_null\n  - name: employee_id\n    description: Employee id.\n  - name: last_name\n    description: Last name.\n  - name: first_name\n    description: First name.\n  - name: title\n    description: Title.\n  - name: title_of_courtesy\n    description: Title of courtesy.\n  - name: birth_date\n    description: Birth date.\n  - name: hire_date\n    description: Hire date.\n  - name: address\n    description: Address.\n  - name: city\n    description: City.\n  - name: region\n    description: Region.\n  - name: postal_code\n    description: Postal code.\n  - name: country\n    description: Country.\n  - name: home_phone\n    description: Home phone.\n  - name: extension\n    description: Extension.\n  - name: notes\n    description: Notes.\n  - name: reports_to\n    description: Reports to.\n  - name: photo_path\n    description: Photo path.\n  - name: employee_full_name\n    description: First name followed by last name.\n- name: dim_products\n  description: Northwind dim_products for the required data mart.\n  columns:\n  - name: product_pk\n    description: Primary surrogate key.\n    data_tests:\n    - unique\n    - not_null\n  - name: product_id\n    description: Product id.\n  - name: product_name\n    description: Product name.\n  - name: supplier_id\n    description: Supplier id.\n    data_tests:\n    - not_null\n    - relationships:\n        arguments:\n          to: ref('dim_suppliers')\n          field: supplier_id\n  - name: category_id\n    description: Category id.\n    data_tests:\n    - not_null\n    - relationships:\n        arguments:\n          to: ref('dim_categories')\n          field: category_id\n  - name: quantity_per_unit\n    description: Quantity per unit.\n  - name: unit_price\n    description: Unit price.\n  - name: units_in_stock\n    description: Units in stock.\n  - name: units_on_order\n    description: Units on order.\n  - name: reorder_level\n    description: Reorder level.\n  - name: discontinued\n    description: Discontinued.\n  - name: category_name\n    description: Related product category name.\n  - name: supplier_name\n    description: Related supplier company name.\n- name: dim_suppliers\n  description: Northwind dim_suppliers for the required data mart.\n  columns:\n  - name: supplier_pk\n    description: Primary surrogate key.\n    data_tests:\n    - unique\n    - not_null\n  - name: supplier_id\n    description: Supplier id.\n  - name: company_name\n    description: Company name.\n  - name: contact_name\n    description: Contact name.\n  - name: contact_title\n    description: Contact title.\n  - name: address\n    description: Address.\n  - name: city\n    description: City.\n  - name: region\n    description: Region.\n  - name: postal_code\n    description: Postal code.\n  - name: country\n    description: Country.\n  - name: phone\n    description: Phone.\n  - name: fax\n    description: Fax.\n  - name: home_page\n    description: Home page.\n- name: dim_categories\n  description: Northwind dim_categories for the required data mart.\n  columns:\n  - name: category_pk\n    description: Primary surrogate key.\n    data_tests:\n    - unique\n    - not_null\n  - name: category_id\n    description: Category id.\n  - name: category_name\n    description: Category name.\n  - name: description\n    description: Description.\n- name: fct_orders\n  description: Northwind fct_orders for the required data mart.\n  columns:\n  - name: order_pk\n    description: One row per order.\n    data_tests:\n    - unique\n    - not_null\n  - name: customer_pk\n    description: Customer dimension key.\n    data_tests:\n    - not_null\n    - relationships:\n        arguments:\n          to: ref('dim_customers')\n          field: customer_pk\n  - name: employee_pk\n    description: Employee dimension key.\n    data_tests:\n    - not_null\n    - relationships:\n        arguments:\n          to: ref('dim_employees')\n          field: employee_pk\n  - name: total_order_value\n    description: Sum of all discounted order-line amounts.\n    data_tests:\n    - not_null\n  - name: order_status_flag\n    description: Shipping status from the shipped timestamp.\n    data_tests:\n    - accepted_values:\n        arguments:\n          values:\n          - Shipped\n          - Pending\n          - Cancelled\n  - name: shipping_delay_days\n    description: Elapsed days between ordered and shipped timestamps; NULL if not\n      shipped.\n- name: fct_order_details\n  description: Northwind fct_order_details for the required data mart.\n  columns:\n  - name: order_detail_pk\n    description: One row per unique order/product pair.\n    data_tests:\n    - unique\n    - not_null\n  - name: order_pk\n    description: Order pk.\n    data_tests:\n    - not_null\n    - relationships:\n        arguments:\n          to: ref('fct_orders')\n          field: order_pk\n  - name: product_pk\n    description: Product pk.\n    data_tests:\n    - not_null\n    - relationships:\n        arguments:\n          to: ref('dim_products')\n          field: product_pk\n  - name: customer_pk\n    description: Customer pk.\n    data_tests:\n    - not_null\n    - relationships:\n        arguments:\n          to: ref('dim_customers')\n          field: customer_pk\n  - name: employee_pk\n    description: Employee pk.\n    data_tests:\n    - not_null\n    - relationships:\n        arguments:\n          to: ref('dim_employees')\n          field: employee_pk\n  - name: order_id\n    description: Order id.\n  - name: product_id\n    description: Product id.\n  - name: unit_price\n    description: Unit price.\n    data_tests:\n    - not_null\n  - name: quantity\n    description: Quantity.\n    data_tests:\n    - not_null\n  - name: discount\n    description: Discount.\n    data_tests:\n    - not_null\n  - name: extended_price\n    description: Extended price.\n    data_tests:\n    - not_null\n  - name: order_date\n    description: Order date.\n    data_tests:\n    - not_null\n  - name: required_date\n    description: Required date.\n  - name: shipped_date\n    description: Shipped date.\n  - name: shipping_delay_days\n    description: Shipping delay days.\n", 'northwind_T/models/staging/schema.yml': "version: 2\nsources:\n- name: staging_dlt\n  schema: staging\n  tables:\n  - name: customers\n  - name: orders\n  - name: employees\n  - name: products\n  - name: suppliers\n  - name: categories\n  - name: order_details\nmodels:\n- name: stg_customers\n  description: Cleaned and typed Northwind customers records.\n  columns:\n  - name: customer_pk\n    description: Stable surrogate key for this record.\n    data_tests:\n    - unique\n    - not_null\n  - name: customer_id\n    description: Customer id, from the original Northwind source.\n    data_tests:\n    - unique\n    - not_null\n  - name: company_name\n    description: Company name, from the original Northwind source.\n  - name: contact_name\n    description: Contact name, from the original Northwind source.\n  - name: contact_title\n    description: Contact title, from the original Northwind source.\n  - name: address\n    description: Address, from the original Northwind source.\n  - name: city\n    description: City, from the original Northwind source.\n  - name: region\n    description: Region, from the original Northwind source.\n  - name: postal_code\n    description: Postal code, from the original Northwind source.\n  - name: country\n    description: Country, from the original Northwind source.\n  - name: phone\n    description: Phone, from the original Northwind source.\n  - name: fax\n    description: Fax, from the original Northwind source.\n- name: stg_orders\n  description: Cleaned and typed Northwind orders records.\n  columns:\n  - name: order_pk\n    description: Stable surrogate key for this record.\n    data_tests:\n    - unique\n    - not_null\n  - name: order_id\n    description: Order id, from the original Northwind source.\n    data_tests:\n    - unique\n    - not_null\n  - name: customer_id\n    description: Customer id, from the original Northwind source.\n    data_tests:\n    - not_null\n    - relationships:\n        arguments:\n          to: ref('stg_customers')\n          field: customer_id\n  - name: employee_id\n    description: Employee id, from the original Northwind source.\n    data_tests:\n    - not_null\n    - relationships:\n        arguments:\n          to: ref('stg_employees')\n          field: employee_id\n  - name: order_date\n    description: Order date, from the original Northwind source.\n    data_tests:\n    - not_null\n  - name: required_date\n    description: Required date, from the original Northwind source.\n  - name: shipped_date\n    description: Shipped date, from the original Northwind source.\n  - name: ship_via\n    description: Ship via, from the original Northwind source.\n  - name: freight\n    description: Freight, from the original Northwind source.\n  - name: ship_name\n    description: Ship name, from the original Northwind source.\n  - name: ship_address\n    description: Ship address, from the original Northwind source.\n  - name: ship_city\n    description: Ship city, from the original Northwind source.\n  - name: ship_region\n    description: Ship region, from the original Northwind source.\n  - name: ship_postal_code\n    description: Ship postal code, from the original Northwind source.\n  - name: ship_country\n    description: Ship country, from the original Northwind source.\n- name: stg_employees\n  description: Cleaned and typed Northwind employees records.\n  columns:\n  - name: employee_pk\n    description: Stable surrogate key for this record.\n    data_tests:\n    - unique\n    - not_null\n  - name: employee_id\n    description: Employee id, from the original Northwind source.\n    data_tests:\n    - unique\n    - not_null\n  - name: last_name\n    description: Last name, from the original Northwind source.\n  - name: first_name\n    description: First name, from the original Northwind source.\n  - name: title\n    description: Title, from the original Northwind source.\n  - name: title_of_courtesy\n    description: Title of courtesy, from the original Northwind source.\n  - name: birth_date\n    description: Birth date, from the original Northwind source.\n  - name: hire_date\n    description: Hire date, from the original Northwind source.\n  - name: address\n    description: Address, from the original Northwind source.\n  - name: city\n    description: City, from the original Northwind source.\n  - name: region\n    description: Region, from the original Northwind source.\n  - name: postal_code\n    description: Postal code, from the original Northwind source.\n  - name: country\n    description: Country, from the original Northwind source.\n  - name: home_phone\n    description: Home phone, from the original Northwind source.\n  - name: extension\n    description: Extension, from the original Northwind source.\n  - name: notes\n    description: Notes, from the original Northwind source.\n  - name: reports_to\n    description: Reports to, from the original Northwind source.\n  - name: photo_path\n    description: Photo path, from the original Northwind source.\n- name: stg_products\n  description: Cleaned and typed Northwind products records.\n  columns:\n  - name: product_pk\n    description: Stable surrogate key for this record.\n    data_tests:\n    - unique\n    - not_null\n  - name: product_id\n    description: Product id, from the original Northwind source.\n    data_tests:\n    - unique\n    - not_null\n  - name: product_name\n    description: Product name, from the original Northwind source.\n  - name: supplier_id\n    description: Supplier id, from the original Northwind source.\n    data_tests:\n    - not_null\n    - relationships:\n        arguments:\n          to: ref('stg_suppliers')\n          field: supplier_id\n  - name: category_id\n    description: Category id, from the original Northwind source.\n    data_tests:\n    - not_null\n    - relationships:\n        arguments:\n          to: ref('stg_categories')\n          field: category_id\n  - name: quantity_per_unit\n    description: Quantity per unit, from the original Northwind source.\n  - name: unit_price\n    description: Unit price, from the original Northwind source.\n  - name: units_in_stock\n    description: Units in stock, from the original Northwind source.\n  - name: units_on_order\n    description: Units on order, from the original Northwind source.\n  - name: reorder_level\n    description: Reorder level, from the original Northwind source.\n  - name: discontinued\n    description: Discontinued, from the original Northwind source.\n- name: stg_suppliers\n  description: Cleaned and typed Northwind suppliers records.\n  columns:\n  - name: supplier_pk\n    description: Stable surrogate key for this record.\n    data_tests:\n    - unique\n    - not_null\n  - name: supplier_id\n    description: Supplier id, from the original Northwind source.\n    data_tests:\n    - unique\n    - not_null\n  - name: company_name\n    description: Company name, from the original Northwind source.\n  - name: contact_name\n    description: Contact name, from the original Northwind source.\n  - name: contact_title\n    description: Contact title, from the original Northwind source.\n  - name: address\n    description: Address, from the original Northwind source.\n  - name: city\n    description: City, from the original Northwind source.\n  - name: region\n    description: Region, from the original Northwind source.\n  - name: postal_code\n    description: Postal code, from the original Northwind source.\n  - name: country\n    description: Country, from the original Northwind source.\n  - name: phone\n    description: Phone, from the original Northwind source.\n  - name: fax\n    description: Fax, from the original Northwind source.\n  - name: home_page\n    description: Home page, from the original Northwind source.\n- name: stg_categories\n  description: Cleaned and typed Northwind categories records.\n  columns:\n  - name: category_pk\n    description: Stable surrogate key for this record.\n    data_tests:\n    - unique\n    - not_null\n  - name: category_id\n    description: Category id, from the original Northwind source.\n    data_tests:\n    - unique\n    - not_null\n  - name: category_name\n    description: Category name, from the original Northwind source.\n  - name: description\n    description: Description, from the original Northwind source.\n- name: stg_order_details\n  description: Cleaned and typed Northwind order_details records.\n  columns:\n  - name: order_detail_pk\n    description: Stable surrogate key for this record.\n    data_tests:\n    - unique\n    - not_null\n  - name: order_id\n    description: Order id, from the original Northwind source.\n    data_tests:\n    - not_null\n    - relationships:\n        arguments:\n          to: ref('stg_orders')\n          field: order_id\n  - name: product_id\n    description: Product id, from the original Northwind source.\n    data_tests:\n    - not_null\n    - relationships:\n        arguments:\n          to: ref('stg_products')\n          field: product_id\n  - name: unit_price\n    description: Unit price, from the original Northwind source.\n  - name: quantity\n    description: Quantity, from the original Northwind source.\n  - name: discount\n    description: Discount, from the original Northwind source.\n", 'northwind_T/models/staging/stg_categories.sql': "select\n    {{ dbt_utils.generate_surrogate_key(['category_id']) }} as category_pk,\n    category_id::bigint as category_id,\n    category_name,\n    description\nfrom {{ source('staging_dlt', 'categories') }}\n", 'northwind_T/models/staging/stg_customers.sql': "select\n    {{ dbt_utils.generate_surrogate_key(['customer_id']) }} as customer_pk,\n    customer_id,\n    company_name,\n    contact_name,\n    contact_title,\n    address,\n    city,\n    region,\n    postal_code,\n    country,\n    phone,\n    fax\nfrom {{ source('staging_dlt', 'customers') }}\n", 'northwind_T/models/staging/stg_employees.sql': "select\n    {{ dbt_utils.generate_surrogate_key(['employee_id']) }} as employee_pk,\n    employee_id::bigint as employee_id,\n    last_name,\n    first_name,\n    title,\n    title_of_courtesy,\n    nullif(birth_date::text, '')::timestamp as birth_date,\n    nullif(hire_date::text, '')::timestamp as hire_date,\n    address,\n    city,\n    region,\n    postal_code,\n    country,\n    home_phone,\n    extension,\n    notes,\n    reports_to::bigint as reports_to,\n    photo_path\nfrom {{ source('staging_dlt', 'employees') }}\n", 'northwind_T/models/staging/stg_orders.sql': "select\n    {{ dbt_utils.generate_surrogate_key(['order_id']) }} as order_pk,\n    order_id::bigint as order_id,\n    customer_id,\n    employee_id::bigint as employee_id,\n    nullif(order_date::text, '')::timestamp as order_date,\n    nullif(required_date::text, '')::timestamp as required_date,\n    nullif(shipped_date::text, '')::timestamp as shipped_date,\n    ship_via::bigint as ship_via,\n    freight::numeric as freight,\n    ship_name,\n    ship_address,\n    ship_city,\n    ship_region,\n    ship_postal_code,\n    ship_country\nfrom {{ source('staging_dlt', 'orders') }}\n", 'northwind_T/models/staging/stg_order_details.sql': "select\n    {{ dbt_utils.generate_surrogate_key(['order_id', 'product_id']) }} as order_detail_pk,\n    order_id::bigint as order_id,\n    product_id::bigint as product_id,\n    unit_price::numeric as unit_price,\n    quantity::bigint as quantity,\n    discount::numeric as discount\nfrom {{ source('staging_dlt', 'order_details') }}\n", 'northwind_T/models/staging/stg_products.sql': "select\n    {{ dbt_utils.generate_surrogate_key(['product_id']) }} as product_pk,\n    product_id::bigint as product_id,\n    product_name,\n    supplier_id::bigint as supplier_id,\n    category_id::bigint as category_id,\n    quantity_per_unit,\n    unit_price::numeric as unit_price,\n    units_in_stock::bigint as units_in_stock,\n    units_on_order::bigint as units_on_order,\n    reorder_level::bigint as reorder_level,\n    discontinued\nfrom {{ source('staging_dlt', 'products') }}\n", 'northwind_T/models/staging/stg_suppliers.sql': "select\n    {{ dbt_utils.generate_surrogate_key(['supplier_id']) }} as supplier_pk,\n    supplier_id::bigint as supplier_id,\n    company_name,\n    contact_name,\n    contact_title,\n    address,\n    city,\n    region,\n    postal_code,\n    country,\n    phone,\n    fax,\n    home_page\nfrom {{ source('staging_dlt', 'suppliers') }}\n"}
for name, content in support_files.items():
    path = Path(name)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content, encoding="utf-8")


## 1. Imports, private connection and source database

In [3]:
import json
import os
from pathlib import Path
import sqlite3

import pandas as pd
import psycopg2
from dbt.cli.main import dbtRunner
from connection import configure_connection
from sqlite_pipeline import TABLES, ensure_database, load_sqlite_data

ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
os.environ["DBT_SEND_ANONYMOUS_USAGE_STATS"] = "false"
url = configure_connection()
with sqlite3.connect(ensure_database()) as source:
    source_counts = {target: source.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
                     for table, target in TABLES.items()}
print("Source row counts:", source_counts)

Source row counts: {'customers': 93, 'orders': 16282, 'employees': 9, 'products': 77, 'categories': 8, 'suppliers': 29, 'order_details': 609283}


## 2. dlt: load the seven complete source tables into Supabase staging

In [4]:
load_sqlite_data()

Loaded: Customers
Loaded: Orders
Loaded: Employees
Loaded: Products
Loaded: Categories
Loaded: Suppliers
Loaded: Order Details


## 3. dbt: connection check, dependencies, staging and marts with their tests

In [5]:
project = ROOT / "northwind_T"
runner = dbtRunner()
dbt_results = []
staging_models = [f"stg_{table}" for table in TABLES.values()]
mart_models = ["dim_customers", "dim_employees", "dim_products", "dim_suppliers",
               "dim_categories", "fct_orders", "fct_order_details"]
commands = [["deps"], ["debug"], ["run", "--select", *staging_models],
            ["test", "--select", *staging_models], ["run", "--select", *mart_models],
            ["test", "--select", *mart_models]]
for command in commands:
    result = runner.invoke(command + ["--project-dir", str(project), "--profiles-dir", str(project)])
    if not result.success:
        raise RuntimeError(f"dbt {' '.join(command)} failed; check the dbt log.") from result.exception
    if command[0] in {"run", "test"} and not result.result.results:
        raise RuntimeError("dbt selected no models or tests; check the selection.")
    dbt_results.append({"command": "dbt " + " ".join(command), "success": True})

20:46:18  Running with dbt=1.12.5
20:46:19  Installing dbt-labs/dbt_utils
20:46:20  Installed from version 1.4.1
20:46:20  Up to date!
20:46:20  Running with dbt=1.12.5
20:46:20  dbt version: 1.12.5
20:46:20  python version: 3.13.7
20:46:20  python path: C:\Python313\python.exe
20:46:20  os info: Windows-11-10.0.26200-SP0
20:46:20  Using profiles dir at C:\Users\chikh\Desktop\UNIV\DataVisualisation-TP2025\DataVisualisation-TP2025--TP6\northwind_T
20:46:20  Using profiles.yml file at C:\Users\chikh\Desktop\UNIV\DataVisualisation-TP2025\DataVisualisation-TP2025--TP6\northwind_T\profiles.yml
20:46:20  Using dbt_project.yml file at C:\Users\chikh\Desktop\UNIV\DataVisualisation-TP2025\DataVisualisation-TP2025--TP6\northwind_T\dbt_project.yml
20:46:20  adapter type: postgres
20:46:20  adapter version: 1.11.0
20:46:20  Configuration:
20:46:20    profiles.yml file [OK found and valid]
20:46:20    dbt_project.yml file [OK found and valid]
20:46:20  Required dependencies:
20:46:20   - git [OK fo

## 4. Verify row counts, fact-table grain and reconciliation to SQLite

In [6]:
warehouse_counts = {}
with psycopg2.connect(url) as connection:
    with connection.cursor() as cursor:
        for table in TABLES.values():
            cursor.execute(f'SELECT COUNT(*) FROM staging.{table}')
            count = cursor.fetchone()[0]
            assert count == source_counts[table], f"Row-count mismatch: {table}"
            warehouse_counts[f"staging.{table}"] = count
        for table in ["dim_customers", "dim_employees", "dim_products", "dim_suppliers",
                      "dim_categories", "fct_orders", "fct_order_details"]:
            cursor.execute(f'SELECT COUNT(*) FROM northwind_warehouse.{table}')
            warehouse_counts[f"northwind_warehouse.{table}"] = cursor.fetchone()[0]
        cursor.execute("""SELECT SUM(extended_price), COUNT(DISTINCT order_id),
                          SUM(quantity), COUNT(*) FROM northwind_warehouse.fct_order_details""")
        revenue, orders, quantity, details = cursor.fetchone()
        cursor.execute("""SELECT order_detail_pk, order_id, product_id, quantity,
                          unit_price, discount, extended_price, order_date, shipping_delay_days
                          FROM northwind_warehouse.fct_order_details
                          ORDER BY order_id, product_id LIMIT 10""")
        preview = pd.DataFrame(cursor.fetchall(), columns=[c.name for c in cursor.description])
with sqlite3.connect(ensure_database()) as source:
    source_revenue, source_quantity, source_orders = source.execute(
        'SELECT SUM(UnitPrice * Quantity * (1 - Discount)), SUM(Quantity), COUNT(DISTINCT OrderID) FROM "Order Details"'
    ).fetchone()
assert abs(float(revenue) - source_revenue) < 0.01
assert int(quantity) == source_quantity and orders == source_orders
assert details == source_counts["order_details"]
summary = {"source_counts": source_counts, "warehouse_counts": warehouse_counts,
           "total_revenue": float(revenue), "total_orders": orders,
           "total_quantity": int(quantity), "detail_rows": details,
           "average_order_value": float(revenue) / orders, "dbt_commands": dbt_results}
(OUT / "verification.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
preview.to_csv(OUT / "fct_order_details_preview.csv", index=False)
print(pd.Series(warehouse_counts).to_string())
print(preview.to_string(index=False))
print("Revenue:", revenue, "Orders:", orders, "Quantity:", quantity)

staging.customers                            93
staging.orders                            16282
staging.employees                             9
staging.products                             77
staging.categories                            8
staging.suppliers                            29
staging.order_details                    609283
northwind_warehouse.dim_customers            93
northwind_warehouse.dim_employees             9
northwind_warehouse.dim_products             77
northwind_warehouse.dim_suppliers            29
northwind_warehouse.dim_categories            8
northwind_warehouse.fct_orders            16282
northwind_warehouse.fct_order_details    609283
                 order_detail_pk  order_id  product_id  quantity unit_price discount    extended_price order_date shipping_delay_days
8c07f317a20b972b5224016d45f5ac7f     10248          11        12  14.000000 0.000000  168.000000000000 2016-07-04 12.0000000000000000
50b71662431db9ba522780ff57fd0cc0     10248          42      

## 5. Required warehouse explanation and documented starter corrections

In [7]:
report = f"""# TP6 — Northwind data mart

All seven instructor-supplied SQLite tables were loaded with dlt into the
Supabase PostgreSQL `staging` schema. Source and destination row counts match.
This source contains **{orders:,} orders** and **{details:,} order-detail rows**;
it is the larger database supplied by the teacher, not a tiny sample.
Suppliers are loaded too, because their staging and dimension models are required.

Seven staging models standardize columns, types and surrogate keys.
Five dimensions describe customers, employees, products, suppliers and categories.
`fct_orders` has one row per order; `fct_order_details` has one row per
order/product pair and keys referencing the dimensions, order dates and measures.
Date fields remain attributes, matching the detailed tasks; no extra date-dimension
or unrelated feature is introduced. Product dimensions include category names;
employee dimensions include full names; customers include first-order dates and
full addresses. Both fact tables include shipping delay, which TP7 needs.

Revenue = unit_price × quantity × (1 − discount).
Verified total revenue: **{float(revenue):,.4f}**; quantity: **{int(quantity):,}**;
average order value: **{float(revenue)/orders:,.2f}**.
SQL uses decimal arithmetic; the SQLite floating-point total agrees within one cent.
These are values in the source's monetary unit; the SQLite data do not establish
a currency, so the reports do not invent one.

All required dbt connection, run and test commands completed successfully.
Primary keys have unique/not-null tests, foreign keys have relationship tests,
and order status is checked against the allowed statuses. Dates lacking shipping
information stay NULL rather than being converted to a zero-day delivery.

Starter corrections: dlt actually normalizes identifiers to snake_case;
dbt references those names. `dbt_utils.generate_surrogate_key` replaces the
removed older `surrogate_key` macro. PostgreSQL timestamp subtraction with
EXTRACT computes shipping days; SQLite-only `julianday` cannot run here.
Prices, freight and discounts have explicit decimal hints during ingestion,
preventing SQLite's mixed integer/float values from splitting into variant columns.
The order fact explicitly includes its customer/employee surrogate keys, so
relationship tests check real columns. Models use views for staging and tables
for marts; rerunning replaces the coursework snapshot instead of duplicating it.
Tests use dbt's built-in assertions, with explicit SQL types, avoiding a second
dependency merely to assert the declared monetary column type.
Model names are selected explicitly because unqualified wildcard selections
do not match folder-qualified models consistently in the installed dbt version.

Credentials come from an ignored `.env` file or hidden input. None are saved
in the notebook or GitHub files. TP7 can now connect to northwind_warehouse.

Assignment: https://github.com/ahmed3991/DataVisualisation-TP2025/blob/main/TP6/TP6.md
"""
(OUT / "REPORT.md").write_text(report, encoding="utf-8")
print("Saved verification results and the TP6 explanation.")

Saved verification results and the TP6 explanation.


## Required written explanation

In [8]:
from IPython.display import display, Markdown
display(Markdown((OUT / "REPORT.md").read_text(encoding="utf-8")))

# TP6 — Northwind data mart

All seven instructor-supplied SQLite tables were loaded with dlt into the
Supabase PostgreSQL `staging` schema. Source and destination row counts match.
This source contains **16,282 orders** and **609,283 order-detail rows**;
it is the larger database supplied by the teacher, not a tiny sample.
Suppliers are loaded too, because their staging and dimension models are required.

Seven staging models standardize columns, types and surrogate keys.
Five dimensions describe customers, employees, products, suppliers and categories.
`fct_orders` has one row per order; `fct_order_details` has one row per
order/product pair and keys referencing the dimensions, order dates and measures.
Date fields remain attributes, matching the detailed tasks; no extra date-dimension
or unrelated feature is introduced. Product dimensions include category names;
employee dimensions include full names; customers include first-order dates and
full addresses. Both fact tables include shipping delay, which TP7 needs.

Revenue = unit_price × quantity × (1 − discount).
Verified total revenue: **448,386,633.1695**; quantity: **15,538,602**;
average order value: **27,538.79**.
SQL uses decimal arithmetic; the SQLite floating-point total agrees within one cent.
These are values in the source's monetary unit; the SQLite data do not establish
a currency, so the reports do not invent one.

All required dbt connection, run and test commands completed successfully.
Primary keys have unique/not-null tests, foreign keys have relationship tests,
and order status is checked against the allowed statuses. Dates lacking shipping
information stay NULL rather than being converted to a zero-day delivery.

Starter corrections: dlt actually normalizes identifiers to snake_case;
dbt references those names. `dbt_utils.generate_surrogate_key` replaces the
removed older `surrogate_key` macro. PostgreSQL timestamp subtraction with
EXTRACT computes shipping days; SQLite-only `julianday` cannot run here.
Prices, freight and discounts have explicit decimal hints during ingestion,
preventing SQLite's mixed integer/float values from splitting into variant columns.
The order fact explicitly includes its customer/employee surrogate keys, so
relationship tests check real columns. Models use views for staging and tables
for marts; rerunning replaces the coursework snapshot instead of duplicating it.
Tests use dbt's built-in assertions, with explicit SQL types, avoiding a second
dependency merely to assert the declared monetary column type.
Model names are selected explicitly because unqualified wildcard selections
do not match folder-qualified models consistently in the installed dbt version.

Credentials come from an ignored `.env` file or hidden input. None are saved
in the notebook or GitHub files. TP7 can now connect to northwind_warehouse.

Assignment: https://github.com/ahmed3991/DataVisualisation-TP2025/blob/main/TP6/TP6.md
